## Step 2 of the seminar

starting from the meta-algorithm, we start going in deep performing a Q-learning. The program still uses 1 agent and one principal

The MDP remains the same (principla_agent_mdp.py), but the optimizations use new implementations

In [1]:
# meta algorithm with Q_leraning
import numpy as np
from principal_agent_mdp import PrincipalAgentMDP
from agent_qlearn import Agent
from principal_isa import Principal

In [ ]:


GAMMA      = 1.0
N_META     = 3
N_EPISODES = 5000
ALPHA      = 0.1
EPSILON    = 0.1

mdp   = PrincipalAgentMDP(gamma=GAMMA)
agent = Agent(mdp, alpha=ALPHA, epsilon=EPSILON)

r_p = np.zeros((mdp.n_states, mdp.n_outcomes))
for s in range(mdp.n_states):
    r_p[s, 0] = 14/9
    r_p[s, 1] = 0.0

principal = Principal(mdp, r_p, alpha=ALPHA, epsilon=EPSILON)

# rho: s -> tuple (b(L), b(R)) — matches Principal output format
rho = {s: (0.0, 0.0) for s in range(mdp.n_states)}

for meta in range(N_META):

    # ── Inner: agent learns — Q_bar changes every step ─────────
    agent.reset()
    for episode in range(N_EPISODES):
        s    = mdp.s0
        while not mdp.is_terminal(s): #process goes on until a final state is reached
            b      = rho[s]
            a      = agent.act(s, b)
            o      = mdp.sample_outcome(s, a)
            s_next = mdp.T(s, o)
            b_next = rho[s_next]

            agent.update(s, a, o, s_next, b_next)  # Q_bar updates here

            s = s_next

    print(f"Q_bar:\n{agent.Q_bar}")

    # ── Outer: Q_bar now fixed — pre-solve LP once ──────────────
    principal.reset()
    Q_bar = agent.get_Q_bar()  # snapshot of learned Q_bar

    # 6 LP calls total (3 states × 2 actions) instead of 5000*steps
    contract_table = {}
    for s in range(mdp.n_states):
        for a_p in range(mdp.n_actions):
            contract_table[(s, a_p)] = principal.find_best_contract(s, a_p, Q_bar)

    for episode in range(N_EPISODES):
        s    = mdp.s0
        done = False
        while not done:
            if np.random.rand() < EPSILON:
                a_p = np.random.randint(mdp.n_actions)
            else:
                a_p = int(np.argmax(principal.q[s]))

            b      = contract_table[(s, a_p)]  # O(1) lookup, no LP
            o      = mdp.sample_outcome(s, a_p)
            s_next = mdp.T(s, o)

            principal.update(s, a_p, b, o, s_next)

            while not mdp.is_terminal(s):
                s = s_next

    # extract rho
    for s in range(mdp.n_states):
        best_a_p = int(np.argmax(principal.q[s]))
        rho[s]   = contract_table[(s, best_a_p)]

    print(f"Principal q:\n{principal.q}")
    for s, name in enumerate(['s0', 'sL', 'sR']):
        print(f"rho({name}) = {rho[s]}")

Q_bar:
[[-0.8  0. ]
 [ 0.   0. ]
 [ 0.   0. ]]
Principal q:
[[380.08844076 356.58370727]
 [380.62299759 360.9403888 ]
 [374.44909528 251.9522312 ]]
rho(s0) = (np.float64(1.0), np.float64(0.0))
rho(sL) = (np.float64(0.0), np.float64(0.0))
rho(sR) = (np.float64(0.0), np.float64(0.0))
Q_bar:
[[-0.8  0. ]
 [ 0.   0. ]
 [ 0.   0. ]]
Principal q:
[[377.5339754  361.40995434]
 [378.36903756 356.20606793]
 [374.21225972 228.66564631]]
rho(s0) = (np.float64(1.0), np.float64(0.0))
rho(sL) = (np.float64(0.0), np.float64(0.0))
rho(sR) = (np.float64(0.0), np.float64(0.0))
Q_bar:
[[-0.8  0. ]
 [ 0.   0. ]
 [ 0.   0. ]]
Principal q:
[[371.16150097 355.29647087]
 [372.17875296 352.96203327]
 [366.80844105 265.4844946 ]]
rho(s0) = (np.float64(1.0), np.float64(0.0))
rho(sL) = (np.float64(0.0), np.float64(0.0))
rho(sR) = (np.float64(0.0), np.float64(0.0))
